In [ ]:
%pip install torch torchvision pillow transformers fastapi

- Standard: ZhengPeng7/BiRefNet (High accuracy)
- Lite: ZhengPeng7/BiRefNet_lite (Faster, smaller)
- Portrait: ZhengPeng7/BiRefNet_portrait (Specialized for people)

In [9]:
import torch
from PIL import Image
from torchvision import transforms
from transformers import AutoModelForImageSegmentation

# 1. Load the model (Automatically downloads from Hugging Face)
model_name = "ZhengPeng7/BiRefNet_lite" 
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForImageSegmentation.from_pretrained(model_name, trust_remote_code=True)
model.to(device)
model.eval()

# 2. Prepare the Image
def bg_image_remover(image_path):
    input_image = Image.open(image_path).convert("RGB")
    
    # Standard BiRefNet normalization
    transform = transforms.Compose([
        transforms.Resize((1024, 1024)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    input_tensor = transform(input_image).unsqueeze(0).to(device)
    
    # Convert input to match model's dtype
    input_tensor = input_tensor.to(next(model.parameters()).dtype)
    
    # 3. Inference
    with torch.no_grad():
        preds = model(input_tensor)[-1].sigmoid().cpu()
        pred = preds[0].squeeze()
    
    # 4. Create Mask and Apply
    mask = transforms.ToPILImage()(pred)
    mask = mask.resize(input_image.size)
    
    input_image.putalpha(mask) # Adds transparency based on the mask
    return input_image


Loading weights: 100%|██████████| 586/586 [00:00<00:00, 2209.36it/s]


In [ ]:
# Run it
result = bg_image_remover("8865880.jpg")
result.save("output_no_bg.png")

# WaterMark Remover

In [1]:
%pip install simple-lama-inpainting

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 3.2 MB/s  0:00:04 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... error
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [20 lines of output]
      + /home/yash/Documents/projects/quickbg/worker/.venv/bin/python /tmp/pip-install-a_hpxn_m/numpy_b9f1b88851a941a38b6bd51e212a53f6/vendored-meson/meson/meson.py setup /tmp/pip-install-a_hpxn_m/numpy_b9f1b88851a941a38b6bd51e212a53f6 /tmp/pip-install-a_hpxn_m/numpy_b9f1b88851a941a38b6bd51e212a53f6/.mesonpy-hvfanupq -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=/tmp/pip-install-a_hpxn_m/numpy_b9f1b88851a941a38b6bd51e21

In [ ]:
from simple_lama_inpainting import SimpleLama
from PIL import Image

# Initialize (This will download the weights on the first run ~200MB)
simple_lama = SimpleLama()

# Load your image and the mask (The mask you'd get from BiRefNet)
# In the mask: White (255) = Area to remove, Black (0) = Keep
image = Image.open("your_image.jpg")
mask = Image.open("your_mask.png").convert('L') 

# Run Inpainting
result = simple_lama(image, mask)

# Save the clean image
result.save("cleaned_image.png")
print("Watermark removed successfully!")

In [ ]:
import torch
from PIL import Image
from torchvision import transforms
from transformers import AutoModelForImageSegmentation
from simple_lama_inpainting import SimpleLama

# 1. Load Models (Optimized for 4GB RAM)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForImageSegmentation.from_pretrained("ZhengPeng7/BiRefNet_lite", trust_remote_code=True)
model.to(device).eval()

simple_lama = SimpleLama()

def remove_watermark_from_frontend(original_img_path, coords, output_path):
    """
    coords: dict like {'x': 100, 'y': 150, 'w': 200, 'h': 50} 
    derived from the user drawing over the watermark in the front end.
    """
    # Load Original
    img = Image.open(original_img_path).convert("RGB")
    original_width, original_height = img.size

    # 1. Define the Box (The area the user 'painted')
    x, y, w, h = coords['x'], coords['y'], coords['w'], coords['h']
    
    # Optional: Add a small padding (10px) to ensure we get the edges of the logo
    padding = 10
    box = (
        max(0, x - padding), 
        max(0, y - padding), 
        min(original_width, x + w + padding), 
        min(original_height, y + h + padding)
    )

    # 2. Crop for BiRefNet (Efficiency)
    roi = img.crop(box)

    # 3. BiRefNet Masking
    transform = transforms.Compose([
        transforms.Resize((1024, 1024)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    input_tensor = transform(roi).unsqueeze(0).to(device)
    input_tensor = input_tensor.to(next(model.parameters()).dtype)
    
    with torch.no_grad():
        preds = model(input_tensor)[-1].sigmoid().cpu()
        mask_pred = preds[0].squeeze()
    
    # 4. Process the Mask
    mask_roi = transforms.ToPILImage()(mask_pred)
    mask_roi = mask_roi.resize(roi.size)
    mask_roi = mask_roi.point(lambda p: 255 if p > 100 else 0) # Hard edges

    # Create full canvas mask
    full_mask = Image.new("L", img.size, 0)
    full_mask.paste(mask_roi, (box[0], box[1]))

    # 5. LaMa Inpaint
    # LaMa uses the area under the 'white' part of full_mask to rebuild the background
    final_img = simple_lama(img, full_mask)
    
    final_img.save(output_path)
    print(f"Clean image saved to {output_path}")


# These coordinates would come from your React/Next.js canvas
frontend_coords = {'x': 850, 'y': 900, 'w': 150, 'h': 60} 

remove_watermark_from_frontend("gemini.png", frontend_coords, "cleaned.png")